# Live Screener — Alpha Picks

Investment decision support notebook. Applies the full screening pipeline:
- Quality gates (Piotroski, Beneish, Altman)
- Margin-of-safety gates (P/B, FCF yield)
- Institution-avoidance niche (micro/small-cap, international)
- IC-weighted composite alpha score
- Kelly-sized long/short recommendations
- Low-vol leverage flag (β-based proxy)

**Run this notebook after the weekly data refresh.**

## 0. Config

In [ ]:
from pathlib import Path

# ── Data source ───────────────────────────────────────────────────────────────
# Set USE_HF=True to pull from HuggingFace Hub; False = local parquet
USE_HF = False
HF_REPO = "ekrash718/stock-fraud-screener"
HF_FILENAME = "historical_dataset_clean.parquet"
LOCAL_PARQUET = Path("../data/historical_dataset_clean.parquet")

# ── Screen parameters ─────────────────────────────────────────────────────────
MARKETS = None          # None = all; or list e.g. ['US', 'CA', 'KR', 'JP', 'BR']
HORIZON = "3y"          # '1y' | '3y' | '5y'
TOP_N_LONG = 25         # top long candidates
TOP_N_SHORT = 15        # worst short candidates

# ── Size filter — institution-avoidance niche ─────────────────────────────────
MIN_MARKET_CAP = 10_000_000   # $10M (micro-cap floor; institutions typically skip <$300M)
MAX_MARKET_CAP = 300_000_000  # $300M ceiling to stay in institution-avoidance zone
                               # Set to None to include all sizes

# ── Quality gates (hard filters) ─────────────────────────────────────────────
PIOTROSKI_MIN  = 5       # ≥5 = healthy; raise to 6 for strict
BENEISH_MAX    = -1.78   # < -1.78 = unlikely manipulator (Beneish 1999)
ALTMAN_MIN     = 1.81    # > 1.81 = out of distress zone

# ── Margin of safety ─────────────────────────────────────────────────────────
PB_MAX         = 3.0     # price-to-book ceiling
FCF_YIELD_MIN  = 0.03    # minimum FCF yield (3%)
APPLY_MOS      = True    # set False to see all without valuation gate

# ── Low-volatility leverage flag ──────────────────────────────────────────────
# Uses beta_12m as proxy (volatility_90d is absent from annual filing rows)
BETA_MAX_FOR_LEVERAGE = 0.80   # stocks with beta < 0.80 flagged as leverage candidates
HALF_KELLY_FRACTION   = 0.25   # conservative quarter-Kelly sizing
MAX_LEVERAGE_MULT     = 2.0    # cap leverage at 2×
POSITION_CAP          = 0.05   # max 5% per position
SECTOR_CAP            = 0.40   # max 40% per SIC sector

# ── IC weights (from alpha_registry.json — selected signals only) ─────────────
# Manually synced from data/alpha_registry.json for portability
IC_WEIGHTS = {
    "alpha_value":      0.1521,
    "alpha_quality":    0.0944,
    "alpha_fraud_risk": 0.1989,
    "ml_1y_oof":        0.1534,
    "ml_3y_oof":        0.3352,
    "ml_5y_oof":        0.3901,
}

## 1. Load Dataset

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)

if USE_HF:
    from huggingface_hub import hf_hub_download
    local_path = hf_hub_download(repo_id=HF_REPO, filename=HF_FILENAME, repo_type="dataset")
    df_full = pd.read_parquet(local_path)
    print(f"Loaded from HuggingFace: {df_full.shape}")
else:
    df_full = pd.read_parquet(LOCAL_PARQUET)
    print(f"Loaded local: {df_full.shape}")

print(f"Columns: {df_full.shape[1]}")
print(f"Markets: {df_full['market'].value_counts().to_dict()}")
print(f"Fiscal year range: {df_full['fiscal_year'].min()} – {df_full['fiscal_year'].max()}")

## 2. Universe Filter — Latest Annual Row per Ticker

In [ ]:
# Keep annual filings only
df = df_full[df_full['period_type'] == 'annual'].copy()

# Latest fiscal year per ticker × market
df = (
    df.sort_values('fiscal_year', ascending=False)
      .groupby(['ticker', 'market'], sort=False)
      .first()
      .reset_index()
)

print(f"Latest annual rows: {len(df):,}")
print(f"Markets: {df['market'].value_counts().to_dict()}")
print(f"Fiscal year distribution:\n{df['fiscal_year'].value_counts().sort_index(ascending=False).head(5)}")

# Market filter
if MARKETS is not None:
    df = df[df['market'].isin(MARKETS)]
    print(f"After market filter ({MARKETS}): {len(df):,} rows")

In [ ]:
# ── Diagnostic: key column availability ───────────────────────────────────────
key_cols = [
    'alpha_composite', 'alpha_value', 'alpha_quality', 'alpha_fraud_risk',
    'ml_1y_oof', 'ml_3y_oof', 'ml_5y_oof',
    'piotroski_f_score', 'beneish_m_score', 'altman_z_score',
    'price_to_book', 'fcf_yield', 'market_cap_at_filing', 'beta_12m',
    'roe_volatility_5yr',
]
avail = {c: f"{df[c].notna().mean()*100:.1f}%" if c in df.columns else "MISSING" for c in key_cols}
print("Column availability in latest annual rows:")
for k, v in avail.items():
    flag = " ⚠️" if v == "MISSING" else ""
    print(f"  {k:35} {v}{flag}")

In [ ]:
# ── Display label: ticker (market) since company_name is absent ───────────────
df['label'] = df['ticker'] + ' (' + df['market'] + ')'

# ── Size filter — institution-avoidance niche ─────────────────────────────────
n0 = len(df)
if 'market_cap_at_filing' in df.columns:
    df = df[df['market_cap_at_filing'].notna() & (df['market_cap_at_filing'] >= MIN_MARKET_CAP)]
    if MAX_MARKET_CAP is not None:
        df = df[df['market_cap_at_filing'] <= MAX_MARKET_CAP]
print(f"After size filter (${MIN_MARKET_CAP/1e6:.0f}M – ${(MAX_MARKET_CAP or 0)/1e6:.0f}M): {len(df):,} / {n0:,} rows")

# ── Quality gates ─────────────────────────────────────────────────────────────
n1 = len(df)
if 'piotroski_f_score' in df.columns:
    df = df[df['piotroski_f_score'] >= PIOTROSKI_MIN]
if 'beneish_m_score' in df.columns:
    df = df[df['beneish_m_score'] < BENEISH_MAX]
if 'altman_z_score' in df.columns:
    df = df[df['altman_z_score'] > ALTMAN_MIN]
print(f"After quality gates (Piotroski≥{PIOTROSKI_MIN}, Beneish<{BENEISH_MAX}, Altman>{ALTMAN_MIN}): {len(df):,} / {n1:,} rows")

# ── Margin of safety ─────────────────────────────────────────────────────────
n2 = len(df)
if APPLY_MOS:
    mos_mask = pd.Series(True, index=df.index)
    if 'price_to_book' in df.columns:
        mos_mask &= (df['price_to_book'].isna() | (df['price_to_book'] <= PB_MAX))
    if 'fcf_yield' in df.columns:
        mos_mask &= (df['fcf_yield'].isna() | (df['fcf_yield'] >= FCF_YIELD_MIN))
    df = df[mos_mask]
    print(f"After margin-of-safety (P/B≤{PB_MAX}, FCF≥{FCF_YIELD_MIN*100:.0f}%): {len(df):,} / {n2:,} rows")

print(f"\nScreened universe: {len(df):,} stocks")

## 3. Composite Alpha Score

In [ ]:
def compute_composite(df: pd.DataFrame, ic_weights: dict) -> pd.Series:
    """IC-weighted composite: percentile-rank each signal then weight-average."""
    ranks = {}
    for col in ic_weights:
        if col in df.columns:
            ranks[col] = df[col].rank(pct=True)

    composite = pd.Series(0.0, index=df.index)
    weight_sum = pd.Series(0.0, index=df.index)
    for col, w in ic_weights.items():
        if col in ranks:
            valid = ranks[col].notna()
            composite[valid] += ranks[col][valid] * w
            weight_sum[valid] += w

    return composite / weight_sum.clip(lower=1e-9)


df['composite_score'] = compute_composite(df, IC_WEIGHTS)

print(f"Composite score — mean: {df['composite_score'].mean():.3f}  "
      f"std: {df['composite_score'].std():.3f}  "
      f"p75: {df['composite_score'].quantile(0.75):.3f}  "
      f"p90: {df['composite_score'].quantile(0.90):.3f}")

In [ ]:
# ── Low-vol leverage flag (beta_12m proxy; volatility_90d absent from annual rows) ──
if 'beta_12m' in df.columns:
    df['leverage_candidate'] = (
        df['beta_12m'].notna() & (df['beta_12m'] < BETA_MAX_FOR_LEVERAGE)
    )
    print(f"Leverage candidates (β < {BETA_MAX_FOR_LEVERAGE}): {df['leverage_candidate'].sum():,}")
else:
    df['leverage_candidate'] = False
    print("beta_12m not available — leverage_candidate set to False")

In [ ]:
# ── Kelly position sizing ─────────────────────────────────────────────────────
def kelly_weights(scores: pd.Series, fraction: float, position_cap: float,
                  sector_col: pd.Series = None, sector_cap: float = 0.40) -> pd.Series:
    f_full = (2.0 * scores - 1.0).clip(lower=0.0)
    f = (f_full * fraction).clip(upper=position_cap)
    # Sector cap
    if sector_col is not None:
        for sic in sector_col.unique():
            mask = (sector_col == sic)
            sw = f[mask].sum()
            if sw > sector_cap:
                f[mask] *= sector_cap / sw
    total = f.sum()
    return f / total if total > 0 else f


sector_col = df['sic_code'] if 'sic_code' in df.columns else None
df['kelly_weight'] = kelly_weights(
    df['composite_score'], HALF_KELLY_FRACTION, POSITION_CAP,
    sector_col, SECTOR_CAP
)
df['kelly_pct'] = (df['kelly_weight'] * 100).round(2)

## 4. Long Picks — Top Candidates

In [ ]:
display_cols = [
    'label', 'fiscal_year', 'market',
    'composite_score',
    'alpha_value', 'alpha_quality', 'alpha_fraud_risk',
    'ml_3y_oof',
    'piotroski_f_score', 'beneish_m_score', 'altman_z_score',
    'price_to_book', 'fcf_yield',
    'market_cap_at_filing', 'beta_12m',
    'leverage_candidate', 'kelly_pct',
]
present = [c for c in display_cols if c in df.columns]

longs = (
    df.nlargest(TOP_N_LONG, 'composite_score')[present]
      .reset_index(drop=True)
)

# Format market cap as $M
if 'market_cap_at_filing' in longs.columns:
    longs['mktcap_$M'] = (longs['market_cap_at_filing'] / 1e6).round(1)
    longs = longs.drop(columns=['market_cap_at_filing'])

longs.index = range(1, len(longs) + 1)
print(f"Top {TOP_N_LONG} LONG candidates:")
longs

In [ ]:
# ── Leverage-eligible longs ───────────────────────────────────────────────────
if df['leverage_candidate'].any():
    lev = (
        df[df['leverage_candidate']]
          .nlargest(min(10, len(df[df['leverage_candidate']])), 'composite_score')[present]
          .reset_index(drop=True)
    )
    if 'market_cap_at_filing' in lev.columns:
        lev['mktcap_$M'] = (lev['market_cap_at_filing'] / 1e6).round(1)
        lev = lev.drop(columns=['market_cap_at_filing'])
    lev.index = range(1, len(lev) + 1)
    print(f"\nTop LEVERAGE candidates (β < {BETA_MAX_FOR_LEVERAGE}, max {MAX_LEVERAGE_MULT}×):")
    display(lev)
else:
    print("No leverage candidates found with current beta threshold.")

## 5. Short Candidates

In [ ]:
# Short candidates: lowest composite score + fraud flags
short_display = [
    'label', 'fiscal_year', 'market',
    'composite_score',
    'alpha_fraud_risk', 'ml_3y_oof',
    'piotroski_f_score', 'beneish_m_score', 'altman_z_score',
    'fraud_suspect', 'montier_c_score',
    'price_to_book', 'market_cap_at_filing',
]
short_present = [c for c in short_display if c in df.columns]

shorts = (
    df.nsmallest(TOP_N_SHORT, 'composite_score')[short_present]
      .reset_index(drop=True)
)
if 'market_cap_at_filing' in shorts.columns:
    shorts['mktcap_$M'] = (shorts['market_cap_at_filing'] / 1e6).round(1)
    shorts = shorts.drop(columns=['market_cap_at_filing'])

shorts.index = range(1, len(shorts) + 1)
print(f"Top {TOP_N_SHORT} SHORT candidates (weakest composite):")
shorts

## 6. Portfolio Summary

In [ ]:
top_longs_df = df.nlargest(TOP_N_LONG, 'composite_score')

print("═" * 50)
print("PORTFOLIO SUMMARY")
print("═" * 50)
print(f"  Screened universe : {len(df):,} stocks")
print(f"  Long picks        : {TOP_N_LONG}")
print(f"  Short picks       : {TOP_N_SHORT}")
print(f"  Horizon           : {HORIZON}")
print()
print("Market breakdown (long picks):")
print(top_longs_df['market'].value_counts().to_string())

print()
print("Score distribution (long picks):")
score_stats = top_longs_df['composite_score'].describe()
print(f"  mean={score_stats['mean']:.3f}  min={score_stats['min']:.3f}  max={score_stats['max']:.3f}")

if 'market_cap_at_filing' in top_longs_df.columns:
    print()
    print("Market cap distribution (long picks, $M):")
    mc = top_longs_df['market_cap_at_filing'] / 1e6
    print(f"  median={mc.median():.1f}  mean={mc.mean():.1f}  min={mc.min():.1f}  max={mc.max():.1f}")

if 'leverage_candidate' in top_longs_df.columns:
    lev_count = top_longs_df['leverage_candidate'].sum()
    print()
    print(f"Leverage-eligible longs: {lev_count} / {TOP_N_LONG} (β < {BETA_MAX_FOR_LEVERAGE})")

print()
print("Quality gate summary (long picks):")
for col, label in [('piotroski_f_score', 'Piotroski mean'),
                   ('beneish_m_score', 'Beneish mean'),
                   ('altman_z_score', 'Altman mean')]:
    if col in top_longs_df.columns:
        print(f"  {label:20} {top_longs_df[col].mean():.2f}")

total_kelly = top_longs_df['kelly_pct'].sum()
print()
print(f"Portfolio allocation (pre-leverage): {total_kelly:.1f}% gross")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Composite score distribution
axes[0].hist(df['composite_score'], bins=40, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(top_longs_df['composite_score'].min(), color='green', linestyle='--', label='Long cutoff')
axes[0].set_title('Composite Score Distribution')
axes[0].set_xlabel('Score')
axes[0].legend()

# Market breakdown
market_counts = top_longs_df['market'].value_counts()
axes[1].bar(market_counts.index, market_counts.values, color='steelblue', edgecolor='black')
axes[1].set_title(f'Long Picks by Market (top {TOP_N_LONG})')
axes[1].set_xlabel('Market')
axes[1].set_ylabel('Count')

# Kelly weight distribution
axes[2].bar(range(TOP_N_LONG), top_longs_df.nlargest(TOP_N_LONG, 'composite_score')['kelly_pct'].values,
            color=['orange' if lc else 'steelblue'
                   for lc in top_longs_df.nlargest(TOP_N_LONG, 'composite_score')['leverage_candidate'].values])
axes[2].set_title('Kelly Weights — Long Picks\n(orange = leverage candidate)')
axes[2].set_xlabel('Rank')
axes[2].set_ylabel('Weight (%)')

plt.tight_layout()
plt.show()

## 7. Export

In [ ]:
from datetime import date
import os

today = date.today().isoformat()
out_dir = Path("../reports")
out_dir.mkdir(exist_ok=True)

long_path  = out_dir / f"screener_longs_{today}.csv"
short_path = out_dir / f"screener_shorts_{today}.csv"

# Prepare export — restore mktcap column
export_cols = [c for c in present if c != 'label']
export_longs = (
    df.nlargest(TOP_N_LONG, 'composite_score')
      [export_cols + ['composite_score', 'kelly_pct', 'leverage_candidate']]
      .reset_index(drop=True)
)
export_shorts = (
    df.nsmallest(TOP_N_SHORT, 'composite_score')
      [export_cols + ['composite_score']]
      .reset_index(drop=True)
)

export_longs.to_csv(long_path, index=False)
export_shorts.to_csv(short_path, index=False)

print(f"Exported {len(export_longs)} longs  → {long_path}")
print(f"Exported {len(export_shorts)} shorts → {short_path}")

---
## Notes

### Low-volatility proxy
`volatility_90d` (from `enrich_market_signals.py`) is absent from annual filing rows — it requires a separate enrichment pass. `beta_12m` is used as a structural proxy: stocks with β < 0.80 carry systematically lower market co-movement and are flagged as leverage candidates. To populate `volatility_90d` run:
```bash
python3 pipeline/enrich_market_signals.py
```

### Company names
`company_name` is not in the parquet — tickers are displayed as `TICKER (MARKET)`. Names can be looked up from `data/tickers_us.csv`, `data/tickers_kr.csv`, etc.

### Leverage sizing
Leverage-eligible stocks receive the same Kelly weight in the output. To apply the actual leverage multiplier:
- Allocate normal Kelly weight from cash
- Borrow additional capital up to `MAX_LEVERAGE_MULT × kelly_weight`
- Maintain drawdown circuit-breaker at 20% portfolio drawdown (see `investment-framework.md` Rule 19)

### Backtest caveat
`data/portfolio_backtest.json` reports 69% CAGR / -0.46% max drawdown — this is a snapshot-based simulation, not a compounded equity curve. The -0.46% drawdown is a clear indicator it measures point-in-time returns, not cumulative wealth. Do not use these numbers for capital allocation decisions.